In [1]:
# ## Best Practises
# 1. Preprocessing and Cleaning (Feature Engineering)
# 2. Train test split
# 3. BOW, TF-IDF, Word2Vec
# 4. Train ML Algorithms

In [2]:
# Loading the dataset
import pandas as pd
data=pd.read_csv("all_kindle_review.csv")
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
# We only require reviewtext and rating
data=data[["reviewText","rating"]]

In [4]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [5]:
data.shape

(12000, 2)

In [6]:
data.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [7]:
data["rating"].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [8]:
data["rating"].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [9]:
## Preprocessing and Cleaning
## I want to predict the sentiment from the review if its positive or negative, hence we can take rating<3 as negative and >=3 as positive
## Positive review = 1 and negative review = 0
data.loc[:,"rating"]=data["rating"].apply(lambda x:0 if x<3 else 1)

In [10]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [11]:
data["rating"].unique()

array([1, 0], dtype=int64)

In [12]:
data["rating"].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [13]:
## 1. Preprocessing Text Data --> 
# 1. LowerCase
data.loc[:,"reviewText"]=data["reviewText"].str.lower()

In [14]:
data.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [15]:
# 2. Clean the data -- I dont want any special characters in this, and stopwords as well
import re
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agarw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
from nltk.corpus import stopwords

In [ ]:
# Remove special character
data.loc[:,"reviewText"]=data["reviewText"].apply(lambda x:re.sub("[^A-Z a-z 0-9-]"," ",x)) # 2 spaces also given, only 1 space was sufficient or could have used r"\s"
# Removing Stopwords
data.loc[:,"reviewText"]=data["reviewText"].apply(lambda x:" ".join([word for word in x.split() if word not in set(stopwords.words("english"))]))
# Remove url
data.loc[:,"reviewText"]=data["reviewText"].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
# Remove html tags

In [18]:
from bs4 import BeautifulSoup
import lxml
data.loc[:,"reviewText"]=data["reviewText"].apply(lambda x:BeautifulSoup(x,"lxml").get_text())
# Remove Additional Spaces
data.loc[:,"reviewText"]=data["reviewText"].apply(lambda x:" ".join(x.split()))

In [19]:
data.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four books expecting 34 con...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [20]:
data.to_csv("Cleaned Reviews.csv",index=True,header=True)

In [21]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(data["reviewText"],data["rating"],test_size=0.20)

In [ ]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

# Cleaner approach will be to reset the index after randomised split
# X_train = X_train.reset_index(drop=True)
# X_test = X_test.reset_index(drop=True)

# y_train = y_train.reset_index(drop=True)
# y_test = y_test.reset_index(drop=True)

((9600,), (2400,), (9600,), (2400,))

In [42]:
X_train[0:2]

1633    liked story part erotica part little better ra...
835     happens serial killer picks serial killer whol...
Name: reviewText, dtype: object

In [23]:
# Training with Word2Vec
from gensim.models import Word2Vec

In [ ]:
# Converting each sentence in the review into list of its words for word2vec training
words=X_train.apply(lambda x:x.split()).values

In [28]:
## Lets train word2vec from scratch according to the dataset
# Word2Vec expects list of words for each sentences and then keeping in the sentence_list, but for BOW, TF-IDF, it was just the sentence column
model=Word2Vec(words)

In [30]:
model.corpus_count,model.epochs

(9600, 5)

In [35]:
model.wv.similar_by_word("good")

[('decent', 0.911137044429779),
 ('ok', 0.9088424444198608),
 ('easy', 0.9058173298835754),
 ('expect', 0.9043775796890259),
 ('enjoyable', 0.8966277241706848),
 ('boring', 0.89442378282547),
 ('okay', 0.8917903900146484),
 ('cute', 0.8914456963539124),
 ('nice', 0.8855947256088257),
 ('entertaining', 0.883231520652771)]

In [36]:
model.wv["good"].shape

(100,)

In [ ]:
import numpy as np
def avg_word_2_vec(doc):
    vectors=[model.wv[word] for word in doc if word in model.wv.index_to_key]
    if(len(vectors)==0):
        return np.zeros(model.vector_size) # If no words (not a single word) from the sentence match with vocab, consider a 0 avg vector for that sentence
    return np.mean(vectors,axis=0)

In [43]:
from tqdm import tqdm
X_train_vector=[]
for review in tqdm(X_train):
    X_train_vector.append(avg_word_2_vec(review))

100%|██████████| 9600/9600 [03:32<00:00, 45.14it/s]


In [44]:
from tqdm import tqdm
X_test_vector=[]
for review in tqdm(X_test):
    X_test_vector.append(avg_word_2_vec(review))

100%|██████████| 2400/2400 [00:51<00:00, 46.71it/s]


In [46]:
X_train_vector=np.array(X_train_vector)
X_test_vector=np.array(X_test_vector)

In [47]:
X_train_vector.shape

(9600, 100)

In [48]:
X_test_vector.shape

(2400, 100)

In [51]:
y_train.shape,y_test.shape

((9600,), (2400,))

In [52]:
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()

In [53]:
gnb.fit(X_train_vector,y_train)

GaussianNB()

In [54]:
y_pred=gnb.predict(X_test_vector)

In [55]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print("Accuracy Score = %.2f"%accuracy_score(y_test,y_pred))
print("Classification_report :")
print(classification_report(y_test,y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test,y_pred))

Accuracy Score = 0.60
Classification_report :
              precision    recall  f1-score   support

           0       0.37      0.29      0.33       800
           1       0.68      0.75      0.71      1600

    accuracy                           0.60      2400
   macro avg       0.52      0.52      0.52      2400
weighted avg       0.58      0.60      0.58      2400

Confusion Matrix
[[ 234  566]
 [ 399 1201]]
